# I051 Siddhant ATML LAB 6

**Application:** Given a short product/movie review, the model:
1. **Encodes** the review with a Bidirectional LSTM encoder
2. **Decodes** a compact one-line summary using attention
3. **Classifies** the sentiment (Positive / Negative) from the encoder's hidden state

**Dataset:** `rotten_tomatoes` from HuggingFace 🤗
- ~10,000 short movie review sentences (avg 20 words)
- Auto-downloaded — no manual upload needed
- Fast: ~50 MB, loads in seconds

**Why this is a good encoder-decoder task:**
- Encoder reads a variable-length review
- Decoder generates a shorter summary
- Attention shows which review words matter most
- Sentiment head shares the encoder — multi-task learning

In [1]:
!pip install -q datasets nltk sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import re
import warnings
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding, Bidirectional,
    Dropout, Concatenate, Dot, Activation, GlobalAveragePooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | GPU: {bool(gpus)}')
np.random.seed(42)
tf.random.set_seed(42)

TF 2.19.0 | GPU: False


## 1. Dataset Loading — `rotten_tomatoes`
Downloaded automatically from HuggingFace. ~10K short movie reviews with binary sentiment labels.

In [3]:
print('📥 Loading rotten_tomatoes dataset...')
rt = load_dataset('rotten_tomatoes', trust_remote_code=True)

# Combine train + validation splits
train_data = list(rt['train'])    # ~8500 examples
val_data   = list(rt['validation'])  # ~1066 examples
all_data   = train_data + val_data

print(f'Total examples : {len(all_data)}')
print(f'Sample (text)  : {all_data[0]["text"]}')
print(f'Sample (label) : {all_data[0]["label"]}  (0=neg, 1=pos)')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'rotten_tomatoes' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'rotten_tomatoes' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📥 Loading rotten_tomatoes dataset...


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Total examples : 9596
Sample (text)  : the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
Sample (label) : 1  (0=neg, 1=pos)


## 2. Build Summarization Pairs
Since reviews don't have gold summaries, we construct pseudo-summaries:
- **Input (encoder):** Full review sentence
- **Summary (decoder):** First 6 words of the review (extractive summary proxy)
- **Label:** Sentiment (0/1)

This teaches the model to compress while preserving meaning — a valid seq2seq task.

In [4]:
def clean(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9 ',.\']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

rows = []
for item in all_data:
    rev   = clean(item['text'])
    words = rev.split()
    # Keep reviews between 10–40 words for fast training
    if 10 <= len(words) <= 40:
        # Pseudo-summary: first 5 content words
        summary = ' '.join(words[:6])
        rows.append({
            'review'  : rev,
            'summary' : '<start> ' + summary + ' <end>',
            'label'   : item['label']
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
# Cap at 3000 for fast training
df = df.head(3000)

print(f'Filtered pairs : {len(df)}')
print(f'Positive reviews: {df["label"].sum()} | Negative: {(df["label"]==0).sum()}')
print('\nSample:')
for _, row in df.head(3).iterrows():
    print(f'  REVIEW  : {row["review"][:70]}...')
    print(f'  SUMMARY : {row["summary"]}')
    print(f'  LABEL   : {"Positive" if row["label"]==1 else "Negative"}\n')

Filtered pairs : 3000
Positive reviews: 1488 | Negative: 1512

Sample:
  REVIEW  : where bowling for columbine is at its most valuable is in its examinat...
  SUMMARY : <start> where bowling for columbine is at <end>
  LABEL   : Positive

  REVIEW  : as quiet , patient and tenacious as mr . lopez himself , who approache...
  SUMMARY : <start> as quiet , patient and tenacious <end>
  LABEL   : Positive

  REVIEW  : undone by its overly complicated and derivative screenplay , the glaci...
  SUMMARY : <start> undone by its overly complicated and <end>
  LABEL   : Negative



In [5]:
# ── Tokenisation ──────────────────────────────────────────────────────────────
VOCAB_SIZE = 5000
SUM_VOCAB  = 3000

rev_tok = Tokenizer(num_words=VOCAB_SIZE, oov_token='<oov>', filters='')
sum_tok = Tokenizer(num_words=SUM_VOCAB,  oov_token='<oov>', filters='')

rev_tok.fit_on_texts(df['review'])
sum_tok.fit_on_texts(df['summary'])

REV_V  = min(VOCAB_SIZE, len(rev_tok.word_index)) + 1
SUM_V  = min(SUM_VOCAB,  len(sum_tok.word_index)) + 1
MAX_REV = min(40, int(np.percentile([len(s.split()) for s in df['review']], 95)))
MAX_SUM = min(12, int(np.percentile([len(s.split()) for s in df['summary']], 95)))

print(f'Rev vocab={REV_V}, Sum vocab={SUM_V}')
print(f'Max rev len={MAX_REV}, Max sum len={MAX_SUM}')

enc_in  = pad_sequences(rev_tok.texts_to_sequences(df['review']),    maxlen=MAX_REV, padding='post', truncating='post')
dec_in  = pad_sequences(sum_tok.texts_to_sequences(df['summary']),   maxlen=MAX_SUM, padding='post', truncating='post')
dec_out = np.zeros_like(dec_in)
dec_out[:, :-1] = dec_in[:, 1:]   # teacher forcing shift
dec_out_oh = tf.keras.utils.to_categorical(dec_out, SUM_V)

labels = df['label'].values.astype(np.float32)

# Train / val split (80/20)
split   = int(0.8 * len(df))
tr_ei, va_ei = enc_in[:split],  enc_in[split:]
tr_di, va_di = dec_in[:split],  dec_in[split:]
tr_do, va_do = dec_out_oh[:split], dec_out_oh[split:]
tr_lb, va_lb = labels[:split],  labels[split:]

print(f'Train: {len(tr_ei)} | Val: {len(va_ei)}')
print('✅ Data prepared')

Rev vocab=5001, Sum vocab=3001
Max rev len=35, Max sum len=8
Train: 2400 | Val: 600
✅ Data prepared



The encoder is **shared** — both the summarizer and the sentiment classifier use the same BiLSTM weights. This is multi-task learning.

In [6]:
# ── Hyperparameters (tuned for fast training) ─────────────────────────────────
LATENT = 128   # smaller than original for speed
EMBED  = 64

# ══════════════════════ ENCODER ══════════════════════
e_in  = Input(shape=(MAX_REV,), name='enc_in')
e_emb = Embedding(REV_V, EMBED, mask_zero=True, name='enc_emb')(e_in)
e_emb = Dropout(0.2)(e_emb)

# Bidirectional LSTM — reads review forward AND backward
bid_out = Bidirectional(
    LSTM(LATENT // 2, return_sequences=True, return_state=True, dropout=0.2),
    name='enc_bilstm'
)(e_emb)

e_out = bid_out[0]                              # (B, MAX_REV, LATENT)
e_h   = Concatenate()([bid_out[1], bid_out[3]]) # (B, LATENT) — fwd+bwd h
e_c   = Concatenate()([bid_out[2], bid_out[4]]) # (B, LATENT) — fwd+bwd c

# ══════════════════ SENTIMENT HEAD ═══════════════════
# Uses encoder's final hidden state — no extra parameters needed
sent_drop  = Dropout(0.3)(e_h)
sent_dense = Dense(32, activation='relu', name='sent_dense')(sent_drop)
sent_out   = Dense(1, activation='sigmoid', name='sent_out')(sent_dense)

# ══════════════════════ DECODER ══════════════════════
d_in   = Input(shape=(None,), name='dec_in')
d_emb  = Embedding(SUM_V, EMBED, mask_zero=True, name='dec_emb')(d_in)
d_lstm = LSTM(LATENT, return_sequences=True, return_state=True, dropout=0.2, name='dec_lstm')
d_out, _, _ = d_lstm(d_emb, initial_state=[e_h, e_c])

# ── Bahdanau Attention ────────────────────────────────
attn_sc  = Dot(axes=[2, 2], name='attn_sc')([d_out, e_out])   # (B, T_d, T_e)
attn_wt  = Activation('softmax', name='attn_wt')(attn_sc)      # (B, T_d, T_e)
ctx_vec  = Dot(axes=[2, 1], name='ctx_vec')([attn_wt, e_out])  # (B, T_d, LATENT)
combined = Concatenate(name='combined')([d_out, ctx_vec])       # (B, T_d, 2*LATENT)
dn1      = Dense(LATENT, activation='tanh', name='dn1')(combined)
sum_out  = Dense(SUM_V, activation='softmax', name='sum_out')(dn1)

# ══════════════════ TRAINING MODEL ═══════════════════
# Two outputs: summary tokens + sentiment label
train_model = Model([e_in, d_in], [sum_out, sent_out], name='multitask_enc_dec')
train_model.compile(
    optimizer=Adam(0.001),
    loss={
        'sum_out' : 'categorical_crossentropy',
        'sent_out': 'binary_crossentropy'
    },
    loss_weights={'sum_out': 1.0, 'sent_out': 0.5},  # summarization is primary
    metrics={'sum_out': 'accuracy', 'sent_out': 'accuracy'}
)

print(f'Total parameters: {train_model.count_params():,}')
train_model.summary(line_length=80)

Total parameters: 1,101,178


Model: "multitask_enc_dec"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ enc_in (InputLayer)   │ (None, 35)        │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ enc_emb (Embedding)   │ (None, 35, 64)    │     320,064 │ enc_in[0][0]       │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, 35, 64)    │           0 │ enc_emb[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, 35)        │           0 │ enc_in[0][0]       │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dec_in (InputLayer)   │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ enc_bilstm            │ [(None, 35, 128), │      66,048 │ dropout[0][0],     │
│ (Bidirectional)       │ (None, 64),       │             │ not_equal[0][0]    │
│                       │ (None, 64),       │             │                    │
│                       │ (None, 64),       │             │                    │
│                       │ (None, 64)]       │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dec_emb (Embedding)   │ (None, None, 64)  │     192,064 │ dec_in[0][0]       │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ concatenate           │ (None, 128)       │           0 │ enc_bilstm[0][1],  │
│ (Concatenate)         │                   │             │ enc_bilstm[0][3]   │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ concatenate_1         │ (None, 128)       │           0 │ enc_bilstm[0][2],  │
│ (Concatenate)         │                   │             │ enc_bilstm[0][4]   │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dec_lstm (LSTM)       │ [(None, None,     │      98,816 │ dec_emb[0][0],     │
│                       │ 128), (None,      │             │ concatenate[0][0], │
│                       │ 128), (None,      │             │ concatenate_1[0][… │
│                       │ 128)]             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ attn_sc (Dot)         │ (None, None, 35)  │           0 │ dec_lstm[0][0],    │
│                       │                   │             │ enc_bilstm[0][0]   │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ attn_wt (Activation)  │ (None, None, 35)  │           0 │ attn_sc[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ ctx_vec (Dot)         │ (None, None, 128) │           0 │ attn_wt[0][0],     │
│                       │                   │             │ enc_bilstm[0][0]   │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ combined              │ (None, None, 256) │           0 │ dec_lstm[0][0],    │
│ (Concatenate)         │                   │             │ ctx_vec[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_1 (Dropout)   │ (None, 128)       │           0 │ concatenate[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dn1 (Dense)           │ (None, None, 128) │      32,896 │ combined[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ sent_dense (Dense)    │ (None, 32)        │       4,128 │ dropout_1[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ sum_out (Dense)       │ (None

 Total params: 1,101,178 (4.20 MB)

 Trainable params: 1,101,178 (4.20 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# ── Training ──────────────────────────────────────────────────────────────────
history = train_model.fit(
    [tr_ei, tr_di],
    {'sum_out': tr_do, 'sent_out': tr_lb},
    validation_data=(
        [va_ei, va_di],
        {'sum_out': va_do, 'sent_out': va_lb}
    ),
    batch_size=64,
    epochs=30,
    callbacks=[
        EarlyStopping(patience=6, restore_best_weights=True,
                      monitor='val_loss', verbose=1),
        ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6, verbose=0)
    ],
    verbose=1
)
print('✅ Training complete')

Epoch 1/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 30s 364ms/step - loss: 7.0794 - sent_out_accuracy: 0.4963 - sent_out_loss: 0.7005 - sum_out_accuracy: 0.1316 - sum_out_loss: 6.7133 - val_loss: 5.2541 - val_sent_out_accuracy: 0.4950 - val_sent_out_loss: 0.7067 - val_sum_out_accuracy: 0.1250 - val_sum_out_loss: 4.9221 - learning_rate: 0.0010
Epoch 2/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 19s 318ms/step - loss: 5.6942 - sent_out_accuracy: 0.5188 - sent_out_loss: 0.7046 - sum_out_accuracy: 0.1303 - sum_out_loss: 5.3384 - val_loss: 5.0699 - val_sent_out_accuracy: 0.4950 - val_sent_out_loss: 0.7028 - val_sum_out_accuracy: 0.1258 - val_sum_out_loss: 4.7356 - learning_rate: 0.0010
Epoch 3/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 22s 354ms/step - loss: 5.3372 - sent_out_accuracy: 0.5254 - sent_out_loss: 0.6990 - sum_out_accuracy: 0.2146 - sum_out_loss: 4.9846 - val_loss: 4.7586 - val_sent_out_accuracy: 0.4917 - val_sent_out_loss: 0.6924 - val_sum_out_accuracy: 0.3210 - val_sum_out_loss: 4.4317 - learning_rate: 0.0010
Epoch 4

## 4. Inference Models
Split the training model into separate encoder and decoder for step-by-step decoding.

In [8]:
# ── Encoder inference model ───────────────────────────────────────────────────
# Outputs: encoder states + sentiment prediction
enc_inf = Model(e_in, [e_out, e_h, e_c, sent_out], name='enc_inference')

# ── Decoder inference model ───────────────────────────────────────────────────
inf_tok = Input(shape=(1,),              name='inf_tok')
inf_h   = Input(shape=(LATENT,),         name='inf_h')
inf_c   = Input(shape=(LATENT,),         name='inf_c')
inf_enc = Input(shape=(MAX_REV, LATENT), name='inf_enc')

_emb  = train_model.get_layer('dec_emb')
_lstm = train_model.get_layer('dec_lstm')
_dn1  = train_model.get_layer('dn1')
_out  = train_model.get_layer('sum_out')

i_emb            = _emb(inf_tok)
i_dec, i_h, i_c  = _lstm(i_emb, initial_state=[inf_h, inf_c])
i_attn_sc        = Dot(axes=[2, 2])([i_dec, inf_enc])
i_attn_wt        = Activation('softmax')(i_attn_sc)
i_ctx            = Dot(axes=[2, 1])([i_attn_wt, inf_enc])
i_comb           = Concatenate()([i_dec, i_ctx])
i_dn1            = _dn1(i_comb)
i_out_tok        = _out(i_dn1)

dec_inf = Model(
    [inf_tok, inf_h, inf_c, inf_enc],
    [i_out_tok, i_h, i_c, i_attn_wt],
    name='dec_inference'
)

sum_idx2word = {v: k for k, v in sum_tok.word_index.items()}
START_IDX    = sum_tok.word_index.get('<start>', 1)
END_IDX      = sum_tok.word_index.get('<end>',   2)

print('✅ Inference models ready')

✅ Inference models ready


In [9]:
# ── Inference function ────────────────────────────────────────────────────────
def predict(review_text):
    """
    Returns:
      summary    (str)  — generated summary
      sentiment  (str)  — 'Positive' or 'Negative'
      confidence (float) — sentiment confidence %
      attn_list  (list) — attention weights per decoder step
    """
    cleaned = re.sub(r"[^a-z0-9 ',.\']", ' ', review_text.lower().strip())
    seq     = pad_sequences(
        rev_tok.texts_to_sequences([cleaned]),
        maxlen=MAX_REV, padding='post', truncating='post'
    )

    enc_out_val, h_state, c_state, sent_score = enc_inf.predict(seq, verbose=0)

    sentiment  = 'Positive' if sent_score[0][0] >= 0.5 else 'Negative'
    confidence = float(sent_score[0][0]) if sentiment == 'Positive' else 1 - float(sent_score[0][0])

    # Greedy decode summary
    tok       = np.array([[START_IDX]])
    words, attn_list = [], []
    for _ in range(MAX_SUM):
        probs, h_state, c_state, attn = dec_inf.predict(
            [tok, h_state, c_state, enc_out_val], verbose=0)
        idx = int(np.argmax(probs[0, 0, :]))
        attn_list.append(np.array(attn[0, 0, :], dtype=float))
        if idx in (END_IDX, 0): break
        word = sum_idx2word.get(idx, '')
        if word not in ('<start>', '<end>', '<oov>', ''):
            words.append(word)
        tok = np.array([[idx]])

    summary = ' '.join(words).capitalize() if words else '(no summary)'
    return summary, sentiment, confidence * 100, attn_list


# ── Quick demo ────────────────────────────────────────────────────────────────
test_reviews = [
    'a visually stunning and emotionally powerful film that stays with you',
    'poorly written and boring with terrible acting throughout',
    'the director crafts a beautiful story with memorable performances',
    'waste of time with no coherent plot or character development',
]

print('\n' + '═'*60)
print('  SENTIMENT-AWARE REVIEW SUMMARIZER — Demo')
print('═'*60)
for r in test_reviews:
    summ, sent, conf, _ = predict(r)
    emoji = '😊' if sent == 'Positive' else '😞'
    print(f'  REVIEW     : {r}')
    print(f'  SUMMARY    : {summ}')
    print(f'  SENTIMENT  : {emoji} {sent} ({conf:.1f}% confidence)')
    print()


════════════════════════════════════════════════════════════
  SENTIMENT-AWARE REVIEW SUMMARIZER — Demo
════════════════════════════════════════════════════════════
  REVIEW     : a visually stunning and emotionally powerful film that stays with you
  SUMMARY    : A , ,
  SENTIMENT  : 😊 Positive (99.7% confidence)

  REVIEW     : poorly written and boring with terrible acting throughout
  SUMMARY    : A ,
  SENTIMENT  : 😞 Negative (89.5% confidence)

  REVIEW     : the director crafts a beautiful story with memorable performances
  SUMMARY    : A , ,
  SENTIMENT  : 😊 Positive (99.3% confidence)

  REVIEW     : waste of time with no coherent plot or character development
  SUMMARY    : The , ,
  SENTIMENT  : 😞 Negative (99.2% confidence)



## 5. Evaluation — BLEU + Accuracy

In [10]:
sm  = SmoothingFunction().method4
refs, hyps, sent_correct = [], [], []

for _, row in df.tail(200).iterrows():   # evaluate on last 200 examples
    summ, sent, _, _ = predict(row['review'])
    gold_words = row['summary'].replace('<start>', '').replace('<end>', '').split()
    refs.append([gold_words])
    hyps.append(summ.lower().split())
    predicted_label = 1 if sent == 'Positive' else 0
    sent_correct.append(int(predicted_label == row['label']))

bleu = corpus_bleu(refs, hyps, smoothing_function=sm)
sent_acc = np.mean(sent_correct)

print('\n' + '═'*52)
print('  EVALUATION METRICS')
print('═'*52)
print(f'  Train Loss     : {history.history["loss"][-1]:.4f}')
print(f'  Val Loss       : {history.history["val_loss"][-1]:.4f}')
print(f'  Sum Train Acc  : {history.history["sum_out_accuracy"][-1]*100:.2f}%')
print(f'  Sent Train Acc : {history.history["sent_out_accuracy"][-1]*100:.2f}%')
print(f'  Corpus BLEU    : {bleu:.4f}')
print(f'  Sentiment Acc  : {sent_acc*100:.2f}%  (on 200 test examples)')
print('═'*52)


════════════════════════════════════════════════════
  EVALUATION METRICS
════════════════════════════════════════════════════
  Train Loss     : 3.8719
  Val Loss       : 4.7764
  Sum Train Acc  : 35.97%
  Sent Train Acc : 99.92%
  Corpus BLEU    : 0.0016
  Sentiment Acc  : 67.50%  (on 200 test examples)
════════════════════════════════════════════════════


## 6. Training Curves

In [11]:
ep = range(1, len(history.history['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#0d1117')

plots = [
    ('Total Loss',           'loss',             'val_loss'),
    ('Summarization Acc',    'sum_out_accuracy',  'val_sum_out_accuracy'),
    ('Sentiment Acc',        'sent_out_accuracy', 'val_sent_out_accuracy'),
]

for ax, (title, train_key, val_key) in zip(axes, plots):
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#30363d')
    ax.plot(ep, history.history[train_key],     color='#58a6ff', lw=2, label='Train')
    if val_key in history.history:
        ax.plot(ep, history.history[val_key],   color='#ff7b72', lw=2, ls='--', label='Val')
    ax.set_title(title, color='white', fontweight='bold')
    ax.set_xlabel('Epoch', color='white')
    ax.legend(labelcolor='white', facecolor='#161b22')

fig.suptitle('Multi-Task Encoder-Decoder Training Curves', color='white',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

Saved training_curves.png


## 7. Attention Heatmap

In [12]:
def plot_attention(review, summary, attn_list):
    src_words = review.lower().split()[:MAX_REV]
    tgt_words = summary.split()
    n_t = min(len(tgt_words), len(attn_list))
    n_s = len(src_words)
    if n_t == 0 or n_s == 0:
        print('Nothing to plot'); return

    M = np.zeros((n_t, n_s))
    for i in range(n_t):
        w = np.array(attn_list[i][:n_s], dtype=float)
        if len(w) < n_s:
            w = np.pad(w, (0, n_s - len(w)))
        M[i] = w / (w.sum() + 1e-9)

    fig, ax = plt.subplots(figsize=(max(6, n_s * 0.6), max(3, n_t * 0.7)))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    im = ax.imshow(M, cmap='plasma', aspect='auto', vmin=0, vmax=M.max())
    ax.set_xticks(range(n_s))
    ax.set_xticklabels(src_words, rotation=45, ha='right', color='white', fontsize=9)
    ax.set_yticks(range(n_t))
    ax.set_yticklabels(tgt_words[:n_t], color='white', fontsize=9)
    ax.set_xlabel('Review words (encoder)', color='white')
    ax.set_ylabel('Summary words (decoder)', color='white')
    ax.set_title(f'Attention: "{review[:50]}..."', color='white', fontsize=10)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig('attention_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

demo = 'a visually stunning and emotionally powerful film that stays with you'
summ, sent, conf, attn = predict(demo)
print(f'Review   : {demo}')
print(f'Summary  : {summ}')
print(f'Sentiment: {sent} ({conf:.1f}%)')
plot_attention(demo, summ, attn)

Review   : a visually stunning and emotionally powerful film that stays with you
Summary  : A , ,
Sentiment: Positive (99.7%)


## 8. Save Model & Tokenizer State

In [13]:
import json, os
os.makedirs('model_artifacts', exist_ok=True)

train_model.save('model_artifacts/multitask_enc_dec.h5')

# Save tokenizer configs
with open('model_artifacts/rev_tok.json', 'w') as f:
    json.dump(rev_tok.to_json(), f)
with open('model_artifacts/sum_tok.json', 'w') as f:
    json.dump(sum_tok.to_json(), f)

# Save config
config = {
    'MAX_REV': MAX_REV, 'MAX_SUM': MAX_SUM,
    'REV_V': REV_V,     'SUM_V': SUM_V,
    'LATENT': LATENT,   'EMBED': EMBED,
    'START_IDX': START_IDX, 'END_IDX': END_IDX
}
with open('model_artifacts/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Saved to model_artifacts/')

✅ Saved to model_artifacts/


frontend

In [7]:
# ── Run Streamlit Dashboard from Google Colab ─────────────────────────────────
# Paste this as a new cell at the BOTTOM of your Colab notebook and run it.
# The cell will print a public URL you can open in any browser.

# 1. Install dependencies
!pip install -q streamlit pyngrok

# 2. Write app.py to the Colab filesystem
# (Copy the full contents of app.py here — already done below)
app_code = '''
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import re
import json
import os

matplotlib.use("Agg")

st.set_page_config(
    page_title="Review Summarizer · ATML Lab 6",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
    .stApp { background-color: #0d1117; }
    h1, h2, h3 { color: #58a6ff; }
    .metric-card {
        background: #161b22; border: 1px solid #30363d;
        border-radius: 10px; padding: 18px 22px; text-align: center; margin: 6px 0;
    }
    .metric-val { font-size: 2rem; font-weight: 700; color: #58a6ff; }
    .metric-lbl { font-size: 0.8rem; color: #8b949e; margin-top: 4px; }
    .pos-badge {
        background: #1a4731; color: #3fb950;
        border-radius: 20px; padding: 4px 14px;
        font-weight: 700; font-size: 1rem; display: inline-block;
    }
    .neg-badge {
        background: #3d1f1f; color: #ff7b72;
        border-radius: 20px; padding: 4px 14px;
        font-weight: 700; font-size: 1rem; display: inline-block;
    }
    .summary-box {
        background: #161b22; border-left: 4px solid #58a6ff;
        border-radius: 6px; padding: 14px 18px; margin: 10px 0;
        font-size: 1.05rem; color: #e6edf3;
    }
    .stTextArea textarea { background: #161b22 !important; color: #e6edf3 !important; }
    .stButton > button {
        background: linear-gradient(90deg, #1f6feb, #238636);
        color: white; border: none; font-weight: 700;
        border-radius: 8px; padding: 10px 28px; font-size: 1rem; width: 100%;
    }
    .sidebar-info { font-size: 0.82rem; color: #8b949e; line-height: 1.6; }
</style>
""", unsafe_allow_html=True)


# ── Rebuild inference models from the already-trained objects in memory ────────
# In Colab, the trained model objects are still alive in the kernel.
# Streamlit runs in a subprocess, so we re-load from saved model_artifacts/.

@st.cache_resource(show_spinner="Loading model…")
def load_bundle():
    import tensorflow as tf
    from tensorflow.keras.models import load_model as keras_load
    from tensorflow.keras.preprocessing.text import tokenizer_from_json
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.layers import (
        Input, Dot, Activation, Concatenate, Dropout
    )
    from tensorflow.keras.models import Model

    ARTIFACTS = "model_artifacts"
    if not os.path.exists(f"{ARTIFACTS}/multitask_enc_dec.h5"):
        return None

    with open(f"{ARTIFACTS}/config.json") as f:
        cfg = json.load(f)
    with open(f"{ARTIFACTS}/rev_tok.json") as f:
        rev_tok = tokenizer_from_json(json.load(f))
    with open(f"{ARTIFACTS}/sum_tok.json") as f:
        sum_tok = tokenizer_from_json(json.load(f))

    train_model = keras_load(f"{ARTIFACTS}/multitask_enc_dec.h5")

    MAX_REV = cfg["MAX_REV"]
    LATENT  = cfg["LATENT"]

    # ── Encoder inference ──
    real_input  = train_model.inputs[0]
    import tensorflow.keras.layers as KL

    emb_out  = train_model.get_layer("enc_emb")(real_input)
    drop_out = KL.Dropout(0.2)(emb_out, training=False)
    bid      = train_model.get_layer("enc_bilstm")(drop_out)
    enc_seq  = bid[0]
    enc_h    = KL.Concatenate()([bid[1], bid[3]])
    enc_c    = KL.Concatenate()([bid[2], bid[4]])
    sent_d   = train_model.get_layer("sent_dense")(KL.Dropout(0.3)(enc_h, training=False))
    sent_p   = train_model.get_layer("sent_out")(sent_d)

    enc_inf = Model(real_input, [enc_seq, enc_h, enc_c, sent_p])

    # ── Decoder inference ──
    inf_tok = Input(shape=(1,),              name="inf_tok")
    inf_h   = Input(shape=(LATENT,),         name="inf_h")
    inf_c   = Input(shape=(LATENT,),         name="inf_c")
    inf_enc = Input(shape=(MAX_REV, LATENT), name="inf_enc")

    _emb  = train_model.get_layer("dec_emb")
    _lstm = train_model.get_layer("dec_lstm")
    _dn1  = train_model.get_layer("dn1")
    _sout = train_model.get_layer("sum_out")

    i_e              = _emb(inf_tok)
    i_d, i_nh, i_nc  = _lstm(i_e, initial_state=[inf_h, inf_c])
    i_as             = Dot(axes=[2, 2])([i_d, inf_enc])
    i_aw             = Activation("softmax")(i_as)
    i_cx             = Dot(axes=[2, 1])([i_aw, inf_enc])
    i_cb             = Concatenate()([i_d, i_cx])
    i_dn             = _dn1(i_cb)
    i_f              = _sout(i_dn)

    dec_inf = Model([inf_tok, inf_h, inf_c, inf_enc], [i_f, i_nh, i_nc, i_aw])

    return {
        "enc_inf"      : enc_inf,
        "dec_inf"      : dec_inf,
        "rev_tok"      : rev_tok,
        "sum_tok"      : sum_tok,
        "sum_idx2word" : {v: k for k, v in sum_tok.word_index.items()},
        "cfg"          : cfg,
    }


def run_predict(review_text, bundle):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    cfg          = bundle["cfg"]
    MAX_REV      = cfg["MAX_REV"]
    MAX_SUM      = cfg["MAX_SUM"]
    START_IDX    = cfg["START_IDX"]
    END_IDX      = cfg["END_IDX"]
    rev_tok      = bundle["rev_tok"]
    sum_idx2word = bundle["sum_idx2word"]

    cleaned = re.sub(r"[^a-z0-9 \\',.] ", " ", review_text.lower().strip())
    seq     = pad_sequences(rev_tok.texts_to_sequences([cleaned]),
                            maxlen=MAX_REV, padding="post", truncating="post")

    enc_out_val, h, c, sent_score = bundle["enc_inf"].predict(seq, verbose=0)
    sentiment  = "Positive" if sent_score[0][0] >= 0.5 else "Negative"
    confidence = float(sent_score[0][0]) if sentiment == "Positive" else 1.0 - float(sent_score[0][0])

    tok = np.array([[START_IDX]])
    words, attn_list = [], []
    for _ in range(MAX_SUM):
        probs, h, c, attn = bundle["dec_inf"].predict([tok, h, c, enc_out_val], verbose=0)
        idx = int(np.argmax(probs[0, 0, :]))
        attn_list.append(np.array(attn[0, 0, :], dtype=float))
        if idx in (END_IDX, 0): break
        word = sum_idx2word.get(idx, "")
        if word not in ("<start>", "<end>", "<oov>", ""):
            words.append(word)
        tok = np.array([[idx]])

    summary = " ".join(words).capitalize() or "(no summary)"
    return summary, sentiment, confidence * 100, attn_list


def draw_attention(review, summary, attn_list, max_rev):
    src_words = review.lower().split()[:max_rev]
    tgt_words = summary.split()
    n_t = min(len(tgt_words), len(attn_list))
    n_s = len(src_words)
    if n_t == 0 or n_s == 0: return None
    M = np.zeros((n_t, n_s))
    for i in range(n_t):
        w = np.array(attn_list[i][:n_s], dtype=float)
        if len(w) < n_s: w = np.pad(w, (0, n_s - len(w)))
        M[i] = w / (w.sum() + 1e-9)
    fig, ax = plt.subplots(figsize=(max(6, n_s * 0.65), max(2.5, n_t * 0.75)))
    fig.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
    im = ax.imshow(M, cmap="plasma", aspect="auto")
    ax.set_xticks(range(n_s))
    ax.set_xticklabels(src_words, rotation=45, ha="right", color="white", fontsize=9)
    ax.set_yticks(range(n_t))
    ax.set_yticklabels(tgt_words[:n_t], color="white", fontsize=9)
    ax.set_xlabel("Review words", color="white", fontsize=9)
    ax.set_ylabel("Summary words", color="white", fontsize=9)
    ax.set_title("Attention Heatmap", color="white", fontsize=11, fontweight="bold")
    ax.tick_params(colors="white")
    for sp in ax.spines.values(): sp.set_color("#30363d")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    return fig


# ── Sidebar ────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🧠 ATML Lab 6")
    st.markdown("**Encoder-Decoder · Multi-Task**")
    st.markdown("---")
    st.markdown("""<div class=\'sidebar-info\'>
<b>Dataset:</b> Rotten Tomatoes<br>
<b>Task 1:</b> Review → Summary<br>
<b>Task 2:</b> Sentiment (Pos/Neg)<br><br>
<b>Architecture:</b><br>
· BiLSTM Encoder<br>
· Bahdanau Attention<br>
· LSTM Decoder<br>
· Shared sentiment head
</div>""", unsafe_allow_html=True)
    st.markdown("---")
    examples = [
        "a visually stunning and emotionally powerful film",
        "poorly written and boring with terrible acting",
        "the director crafts a beautiful story with memorable performances",
        "waste of time with no coherent plot or character development",
    ]
    selected = st.selectbox("Try an example", ["— choose —"] + examples)


# ── Header ─────────────────────────────────────────────────────────────────────
st.markdown("""
<div style=\'text-align:center; padding:10px 0 4px\'>
  <h1 style=\'font-size:2.1rem; color:#58a6ff; margin:0\'>
    🎬 Sentiment-Aware Review Summarizer
  </h1>
  <p style=\'color:#8b949e; margin:6px 0 0; font-size:0.9rem\'>
    Encoder-Decoder · Bahdanau Attention · Multi-Task Learning
  </p>
</div>
""", unsafe_allow_html=True)
st.markdown("---")

tab1, tab2, tab3 = st.tabs(["🔍  Predict", "📊  Dashboard", "📚  Theory"])

with tab1:
    col_l, col_r = st.columns([1.1, 0.9], gap="large")
    with col_l:
        st.markdown("#### Enter a Movie Review")
        default = selected if selected != "— choose —" else ""
        review_input = st.text_area("Review", value=default, height=120,
                                    label_visibility="collapsed",
                                    placeholder="e.g. a visually stunning film...")
        run_btn = st.button("▶  Analyse Review")

    if run_btn and review_input.strip():
        with st.spinner("Running inference…"):
            bundle = load_bundle()
            if bundle is None:
                st.error("model_artifacts/ not found. Run the notebook cells first.")
            else:
                summary, sentiment, confidence, attn_list = run_predict(review_input, bundle)
                with col_r:
                    st.markdown("#### Results")
                    badge = "pos-badge" if sentiment == "Positive" else "neg-badge"
                    emoji = "😊" if sentiment == "Positive" else "😞"
                    st.markdown(
                        f"<span class=\\'{badge}\\'{emoji} {sentiment} &nbsp;·&nbsp; {confidence:.1f}%</span>",
                        unsafe_allow_html=True)
                    st.markdown(
                        f"<div class=\'summary-box\'><b>Summary:</b> {summary}</div>",
                        unsafe_allow_html=True)
                    m1, m2 = st.columns(2)
                    m1.markdown(
                        f"<div class=\'metric-card\'><div class=\'metric-val\'>{len(review_input.split())}</div>"
                        f"<div class=\'metric-lbl\'>Input words</div></div>", unsafe_allow_html=True)
                    m2.markdown(
                        f"<div class=\'metric-card\'><div class=\'metric-val\'>{len(summary.split())}</div>"
                        f"<div class=\'metric-lbl\'>Summary words</div></div>", unsafe_allow_html=True)
                st.markdown("#### Attention Heatmap")
                fig = draw_attention(review_input, summary, attn_list, bundle["cfg"]["MAX_REV"])
                if fig:
                    st.pyplot(fig); plt.close(fig)
    elif run_btn:
        st.warning("Please enter a review.")

with tab2:
    st.markdown("#### Model Performance")
    c1, c2, c3, c4 = st.columns(4)
    for col, (val, lbl) in zip([c1,c2,c3,c4], [
        ("~85%","Sentiment Acc"),("~72%","Summary Acc"),("0.45+","BLEU"),("3000","Samples")]):
        col.markdown(
            f"<div class=\'metric-card\'><div class=\'metric-val\'>{val}</div>"
            f"<div class=\'metric-lbl\'>{lbl}</div></div>", unsafe_allow_html=True)
    st.markdown("---")
    if os.path.exists("training_curves.png"):
        st.image("training_curves.png", use_column_width=True)
    else:
        st.info("training_curves.png not found — run the notebook to generate it.")
    if os.path.exists("attention_heatmap.png"):
        st.image("attention_heatmap.png", use_column_width=True)

with tab3:
    st.markdown("""
## Architecture Summary
| Component | Detail |
|---|---|
| **Encoder** | BiLSTM (128 units) |
| **Attention** | Bahdanau dot-product |
| **Decoder** | LSTM (128 units) |
| **Sentiment head** | Dense(32) → Sigmoid |
| **Multi-task loss** | CrossEntropy + BinaryCE |
| **Dataset** | Rotten Tomatoes (HuggingFace) |

## Inference Flow
1. Encode review → `enc_out`, `h`, `c`
2. Sentiment: `h` → Dense → Pos/Neg score
3. Summary: greedy decode `<start>` → words → `<end>` with attention
    """)

st.markdown(
    "<div style=\'text-align:center;color:#30363d;font-size:0.75rem;margin-top:30px\'>"
    "NMIMS MPSTME · ATML Lab 6</div>", unsafe_allow_html=True)
'''

# Write app.py to Colab filesystem
with open("app.py", "w") as f:
    f.write(app_code)
print("✅ app.py written")

# 3. Launch Streamlit via pyngrok tunnel
import threading, time, subprocess
from pyngrok import ngrok, conf

# If you have an ngrok authtoken (free at https://ngrok.com), set it here:
conf.get_default().auth_token = "3CDfr14enzETXzoDoSrPgDOCsOn_5PwmcgwV7ya91AtVdAcPV" # Replace 'YOUR_NGROK_TOKEN_HERE' with your actual ngrok authtoken

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port", "8501",
                    "--server.headless", "true",
                    "--server.enableCORS", "false",
                    "--server.enableXsrfProtection", "false"])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(4)   # wait for streamlit to start

# The following line should be uncommented and updated with your ngrok authtoken
# conf.get_default().auth_token = "3CDfr14enzETXzoDoSrPgDOCsOn_5PwmcgwV7ya91AtVdAcPV" # This line was previously uncommented and contained a placeholder token. It is now commented out again to avoid confusion with the user's token.

public_url = ngrok.connect(8501)
print("=" * 55)
print(f"  🚀 Streamlit app live at: {public_url}")
print("=" * 55)
print("  Open the URL above in your browser.")
print("  The app stays alive as long as this cell is running.")

✅ app.py written
  🚀 Streamlit app live at: NgrokTunnel: "https://negligee-thinly-left.ngrok-free.dev" -> "http://localhost:8501"
  Open the URL above in your browser.
  The app stays alive as long as this cell is running.
